# Case File 02: The Suspicious Spreadsheet

## Will They Show Up? Campus Event Predictor

The Campus Events Intelligence Unit has received 320 historical event records. Unfortunately, the spreadsheet appears to have been maintained by several people, one hurried robot, and possibly a sandwich.

Your job is to make the evidence trustworthy enough for Module 7. This is a **synthetic dataset**. It describes fictional events and contains no real student information.

**Routine:** Predict, inspect, decide, clean, verify, explain. Never hide a cleaning decision.

## 0. Import the case files into Colab

1. Download `campus_events_raw.csv` and this notebook from Canvas.
2. Open [Google Colab](https://colab.research.google.com/) and upload the notebook.
3. Use the folder icon to upload the CSV into the current session.
4. Confirm the filename before running the next cell.
5. Save a copy of the notebook in Google Drive. Colab session uploads can disappear when the runtime resets.

If you keep the CSV in Drive instead, mount Drive and update `DATA_PATH`. Start with the direct upload method because it has fewer moving parts.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# pandas gives Python tools for working with tables.
import pandas as pd
from pathlib import Path

DATA_PATH = Path('campus_events_raw.csv')

# Stop early with a useful message if the file is missing.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Upload campus_events_raw.csv using Colab's folder panel, then run this cell again."
    )

events = pd.read_csv(DATA_PATH)
print('Case file loaded:', DATA_PATH.name)
events.head()

## 1. Predict before inspecting

Before running the health check, write three problems you think a historical event spreadsheet might contain. Prediction makes inspection purposeful.

1. I predict...
2. I predict...
3. I predict...

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Shape tells us the number of rows and columns.
print('Rows and columns:', events.shape)
print('\nColumns:', events.columns.tolist())
print('\nData types:')
print(events.dtypes)
print('\nMissing values:')
print(events.isna().sum())
print('\nExact duplicate rows:', events.duplicated().sum())
print('Repeated event IDs:', events.duplicated(subset=['event_id']).sum())

## 2. Understand the source before changing values

**Data Source Passport**

- Creator: course team
- Purpose: beginner data-cleaning and machine-learning practice
- Unit of one row: one fictional campus event
- Collection method: generated from transparent classroom rules with controlled randomness
- Privacy: no real people, schools, or events
- License and use: course practice material
- Known limitation: simplified patterns do not represent every reason people attend events

Synthetic does not mean perfect or neutral. The design choices still shape what the later model can learn.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Scan categories exactly as written. Whitespace and capitalization matter.
category_columns = [
    'event_type', 'organizer_type', 'day_of_week', 'venue_type',
    'indoor_outdoor', 'registration_required', 'free_event',
    'food_available', 'food_type', 'prize_available', 'email_campaign',
    'weather_forecast', 'actual_weather', 'campus_activity_level', 'high_turnout'
]
for column in category_columns:
    print(f'\n{column}:')
    print(events[column].value_counts(dropna=False))

# Convert temporary copies to numbers so suspicious text becomes visible as missing.
numeric_columns = [
    'start_hour', 'duration_hours', 'capacity', 'accessibility_score',
    'ticket_price_usd', 'promotion_days', 'promotion_channels', 'social_posts',
    'poster_count', 'competing_events', 'previous_similar_attendance',
    'actual_attendance', 'attendance_rate'
]
for column in numeric_columns:
    numeric_preview = pd.to_numeric(events[column], errors='coerce')
    print(column, 'values that are missing or not numeric:', numeric_preview.isna().sum())

## 3. Create a decision log before cleaning

For each issue, decide whether to correct, remove, replace, or flag it. A decision is only defensible when the rule is stated. The supplied rules below are appropriate for this synthetic practice file, not universal rules for every dataset.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Preserve the original evidence and work on a copy.
clean = events.copy()

# Remove accidental spaces and standardize capitalization.
category_columns = [
    'event_type', 'organizer_type', 'day_of_week', 'venue_type',
    'indoor_outdoor', 'registration_required', 'free_event', 'food_available',
    'food_type', 'prize_available', 'email_campaign', 'weather_forecast',
    'actual_weather', 'campus_activity_level', 'high_turnout'
]
for column in category_columns:
    clean[column] = clean[column].astype(str).str.strip().str.lower()

# Standardize only variations whose meanings are known.
clean['day_of_week'] = clean['day_of_week'].replace({'fri': 'friday'})
clean['actual_weather'] = clean['actual_weather'].replace({'rainy': 'rain'})
for column in ['registration_required', 'free_event', 'food_available',
               'prize_available', 'email_campaign', 'high_turnout']:
    clean[column] = clean[column].replace(
        {'y': 'yes', 'n': 'no', 'true': 'yes', 'false': 'no'}
    )

# Convert readable dates to one consistent format.
clean['event_date'] = pd.to_datetime(clean['event_date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Convert a known number word. Unknown words should not be guessed.
clean['promotion_channels'] = clean['promotion_channels'].replace({'three': 3})

# Remove a currency symbol, then turn invalid text such as 'sandwich' into NaN.
clean['ticket_price_usd'] = clean['ticket_price_usd'].astype(str).str.replace('$', '', regex=False)
numeric_columns = [
    'start_hour', 'duration_hours', 'capacity', 'accessibility_score',
    'ticket_price_usd', 'promotion_days', 'promotion_channels', 'social_posts',
    'poster_count', 'competing_events', 'previous_similar_attendance',
    'actual_attendance', 'attendance_rate'
]
for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors='coerce')

# Remove only exact repeated records. Similar events are not automatically duplicates.
duplicate_count = int(clean.duplicated().sum())
clean = clean.drop_duplicates().copy()

# Flag values that violate the documented rules.
invalid_duration_count = int((~clean['duration_hours'].between(0.5, 12)).sum())
clean.loc[~clean['duration_hours'].between(0.5, 12), 'duration_hours'] = pd.NA
clean.loc[clean['ticket_price_usd'] < 0, 'ticket_price_usd'] = pd.NA
clean.loc[~clean['accessibility_score'].between(1, 5), 'accessibility_score'] = pd.NA
clean.loc[~clean['poster_count'].between(0, 200), 'poster_count'] = pd.NA
clean.loc[clean['actual_attendance'] > clean['capacity'], 'actual_attendance'] = pd.NA

# Use the median only for selected practice columns, and record the limitation.
for column in ['duration_hours', 'ticket_price_usd', 'promotion_days',
               'accessibility_score', 'poster_count']:
    clean[column] = clean[column].fillna(clean[column].median())

# Resolve contradictions using the definitions in the data dictionary.
clean.loc[clean['free_event'] == 'yes', 'ticket_price_usd'] = 0
clean.loc[clean['food_available'] == 'no', 'food_type'] = 'none'

# Estimate missing or impossible attendance with a transparent practice rule.
valid_rates = clean['attendance_rate'].where(clean['attendance_rate'].between(0, 1))
median_rate = valid_rates.median()
missing_attendance = clean['actual_attendance'].isna()
clean.loc[missing_attendance, 'actual_attendance'] = (
    clean.loc[missing_attendance, 'capacity'] * median_rate
).round()

# Recalculate derived outcomes so they agree with the cleaned evidence.
clean['attendance_rate'] = (clean['actual_attendance'] / clean['capacity']).round(3)
clean['high_turnout'] = clean['attendance_rate'].ge(0.65).map({True: 'yes', False: 'no'})

cleaning_log = pd.DataFrame([
    {'issue': 'spacing, capitalization, and dates', 'decision': 'standardized known formats', 'risk': 'unknown meanings must not be guessed'},
    {'issue': 'numeric text and range errors', 'decision': 'converted types and flagged invalid values', 'risk': 'a rule cannot confirm what originally happened'},
    {'issue': 'exact duplicates', 'decision': f'removed {duplicate_count} repeated records', 'risk': 'similar events are not automatically duplicates'},
    {'issue': 'implausible duration', 'decision': f'flagged {invalid_duration_count} value; median filled', 'risk': 'real projects require source confirmation'},
    {'issue': 'free-event and food contradictions', 'decision': 'applied data-dictionary definitions', 'risk': 'the original entry may still need investigation'},
    {'issue': 'missing or impossible attendance', 'decision': 'estimated from median valid attendance rate', 'risk': 'estimated outcomes add uncertainty'},
    {'issue': 'derived outcomes', 'decision': 'recalculated attendance_rate and high_turnout', 'risk': 'the 65 percent threshold is a course design choice'},
])
cleaning_log

## 4. Filter, sort, and summarize without claiming cause

Filtering keeps selected rows. Sorting changes their order. Grouping summarizes many rows. These operations can reveal a pattern, but they cannot prove why it occurred.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Focus on rainy outdoor events and show the least attended first.
focused_events = clean[
    (clean['indoor_outdoor'] == 'outdoor') & (clean['actual_weather'] == 'rain')
].sort_values('actual_attendance')
focused_events[['event_id', 'event_type', 'actual_weather', 'actual_attendance', 'attendance_rate']].head(10)

# Compare turnout by food availability. This is descriptive, not causal.
food_summary = clean.groupby('food_available').agg(
    events=('event_id', 'count'),
    average_attendance=('actual_attendance', 'mean'),
    median_attendance=('actual_attendance', 'median'),
).round(1)
food_summary

## 5. Compare before and after

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
comparison = pd.DataFrame({
    'measure': ['rows', 'missing cells', 'exact duplicates'],
    'raw': [len(events), int(events.isna().sum().sum()), int(events.duplicated().sum())],
    'clean': [len(clean), int(clean.isna().sum().sum()), int(clean.duplicated().sum())],
})
comparison

## 6. Validate the cleaned evidence

Passing these checks means the table follows our stated rules. It does not prove that every historical record is true or that the dataset represents every campus.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
validation_checks = {
    'event_id_unique': clean['event_id'].is_unique,
    'duration_between_0.5_and_12': clean['duration_hours'].between(0.5, 12).all(),
    'price_not_negative': (clean['ticket_price_usd'] >= 0).all(),
    'free_events_cost_zero': clean.loc[clean['free_event'] == 'yes', 'ticket_price_usd'].eq(0).all(),
    'accessibility_between_1_and_5': clean['accessibility_score'].between(1, 5).all(),
    'attendance_not_negative': (clean['actual_attendance'] >= 0).all(),
    'attendance_not_above_capacity': (clean['actual_attendance'] <= clean['capacity']).all(),
    'rate_matches_attendance': ((clean['actual_attendance'] / clean['capacity']).round(3) == clean['attendance_rate']).all(),
    'target_matches_65_percent_rule': (clean['attendance_rate'].ge(0.65).map({True: 'yes', False: 'no'}) == clean['high_turnout']).all(),
    'yes_no_columns_known': all(clean[c].isin(['yes', 'no']).all() for c in ['registration_required', 'free_event', 'food_available', 'prize_available', 'email_campaign', 'high_turnout']),
    'weather_known': clean['actual_weather'].isin(['clear', 'cloudy', 'rain', 'windy']).all(),
}
validation_checks

## 7. Protect Module 8 from target leakage

`high_turnout` is the target. `actual_attendance` and `attendance_rate` directly determine that target, so using either as an input would give the future model the answer. `actual_weather` is also unavailable if the prediction is made before the event. Keep these columns for historical analysis, but exclude them from advance model inputs.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
future_features = [
    'event_type', 'organizer_type', 'day_of_week', 'start_hour',
    'duration_hours', 'venue_type', 'indoor_outdoor', 'capacity',
    'registration_required', 'accessibility_score', 'free_event',
    'ticket_price_usd', 'food_available', 'food_type', 'prize_available',
    'promotion_days', 'promotion_channels', 'social_posts', 'email_campaign',
    'poster_count', 'weather_forecast', 'competing_events',
    'campus_activity_level', 'previous_similar_attendance'
]
target = 'high_turnout'
excluded_from_model = ['event_id', 'event_date', 'actual_weather', 'actual_attendance', 'attendance_rate']
print('Future features:', future_features)
print('Target:', target)
print('Excluded:', excluded_from_model)

## 8. Export the clean handoff file

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
OUTPUT_PATH = 'campus_events_clean.csv'
clean.to_csv(OUTPUT_PATH, index=False)
print('Saved', OUTPUT_PATH, 'with shape', clean.shape)

## 9. Write the evidence note

Complete these statements using your outputs:

- The raw file contained ___ rows and ___ columns.
- The most important quality problems were ___.
- I handled them by ___.
- One descriptive pattern I observed was ___.
- This pattern does not prove ___.
- One limitation cleaning did not solve is ___.

## Required Gemini verification record

Ask Gemini to explain or review one cleaning step. Do not ask it to complete the entire case file.

- My prompt:
- Gemini suggested:
- I tested it by:
- Official pandas documentation I checked:
- I accepted, changed, or rejected the suggestion because:

Never enter private or real student information.

## Module 7 handoff

Keep these together in your Phase 2 project folder:

1. This completed notebook with outputs and explanations
2. `campus_events_clean.csv`
3. The cleaning log and evidence note inside the notebook
4. `campus_events_data_dictionary.csv`

In Module 7, the investigation becomes **Patterns in the Crowd**. You will use statistics and charts to ask what successful events appear to share, while avoiding unsupported causal claims.